# 05 - Tryb JSON

Tryb JSON w Chat Completions OpenAI pozwala deweloperom otrzymywać odpowiedzi z danymi strukturalnymi w formacie JSON. 
Ta funkcja jest szczególnie przydatna w aplikacjach wymagających precyzyjnego parsowania danych i integracji z innymi systemami. Określając schemat JSON, deweloperzy mogą kierować modelem tak, aby generował odpowiedzi zgodne z wcześniej zdefiniowaną strukturą, co zapewnia spójność i wygodę podczas pracy ze złożonymi interakcjami danych. Takie podejście zwiększa użyteczność modelu w scenariuszach takich jak tworzenie API, przetwarzanie danych i automatyczne przepływy pracy, zapewniając bardziej niezawodne i uporządkowane wyniki.

Najpierw używamy instrukcji `import`, aby poinformować aplikację, że w naszym kodzie będziemy korzystać z biblioteki OpenAI.

In [ ]:
%pip install openai

Utworzymy nowy obiekt `AzureOpenAI` i przekażemy do niego klucz API, wersję oraz adres URL punktu końcowego.

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

if load_dotenv():
    print("Znaleziono bazowy punkt końcowy Azure OpenAI API: " + os.getenv("AZURE_OPENAI_ENDPOINT"))
else: 
    print("Nie znaleziono bazowego punktu końcowego Azure OpenAI API. Czy plik .env został skonfigurowany?")
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

def print_response(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),
        messages = [{"role" : "system", "content" : system_prompt}, {"role" : "user", "content" : user_prompt}],
    )
    print(response.choices[0].message.content)

## Wysyłanie promptu do Azure OpenAI przy użyciu biblioteki OpenAI

Skoro zdefiniowaliśmy już instancję Azure OpenAI, spróbujmy wykonać Chat Completion. Wywołamy metodę `chat.completions.create()`. Zwróć uwagę, że jako wartość `model` przekazujemy tutaj identyfikator naszego wdrożenia Azure OpenAI `deployment`. Przekażemy też `prompt`, którego chcemy użyć jako `content` parametru `messages`.

* Ćwiczenie: Napisz prompt, który wygeneruje oczekiwane uzupełnienie
* Tekst wejściowy:
  ```
  Cześć, nazywam się Mateo Gomez. Zgubiłem kartę kredytową 17 sierpnia i chciałbym poprosić o jej anulowanie. Ostatni zakup, którego dokonałem, to danie Chicken parmigiana w Contoso Restaurant, położonej w pobliżu Hollywood Museum, za 40 dolarów. Poniżej podaję moje dane osobowe do weryfikacji:
  Zawód: księgowy
  Numer ubezpieczenia społecznego: 123-45-6789
  Data urodzenia: 9-9-1989
  Numer telefonu: 949-555-0110
  Adres zamieszkania: 1234 Hollywood Boulevard Los Angeles CA
  Powiązane konto e-mail: mateo@contosorestaurant.com
  Kod SWIFT: CHASUS33XXX
  ```
* Oczekiwane uzupełnienie:
  ```
  {
      "reason": "Lost card",
      "classified_reason": "lost_card",
      "name": "Mateo Gomez",
      "ssn": "123-45-6789",
      "dob": "09/09/1989"
  }
  ```

In [ ]:
system_prompt ="""
Mając tekst wejściowy zawierający dane osobowe klienta oraz szczegóły prośby, wyodrębnij i uporządkuj istotne informacje w obiekt JSON. 
Obiekt powinien zawierać powód prośby, sklasyfikowany kod powodu oraz kluczowe dane osobowe klienta. 
"""
user_prompt =""" 
  Cześć, nazywam się Mateo Gomez. Zgubiłem kartę kredytową 17 sierpnia i chciałbym poprosić o jej anulowanie. Ostatni zakup, którego dokonałem, to danie Chicken parmigiana w Contoso Restaurant, położonej w pobliżu Hollywood Museum, za 40 dolarów. Poniżej podaję moje dane osobowe do weryfikacji:
  Zawód: księgowy
  Numer ubezpieczenia społecznego: 123-45-6789
  Data urodzenia: 9-9-1989
  Numer telefonu: 949-555-0110
  Adres zamieszkania: 1234 Hollywood Boulevard Los Angeles CA
  Powiązane konto e-mail: mateo@contosorestaurant.com
  Kod SWIFT: CHASUS33XXX
  """
print_response(system_prompt=system_prompt, user_prompt=user_prompt)

# Czy możemy zrobić to lepiej?

Możemy podać modelowi dodatkowe przykłady lub oczekiwane uzupełnienia, aby poprawić dokładność. 

In [ ]:
system_prompt ="""
Mając tekst wejściowy zawierający dane osobowe klienta oraz szczegóły prośby, wyodrębnij i uporządkuj istotne informacje w obiekt JSON. 
Obiekt powinien zawierać powód prośby, sklasyfikowany kod powodu oraz kluczowe dane osobowe klienta. 

* Oczekiwane uzupełnienie:

  {
      "reason": "Nowe konto",
      "classified_reason": "new_account",
      "name": "John Doe",
      "ssn": "6789",
      "dob": "01/01/2026"
  }

"""
print_response(system_prompt=system_prompt, user_prompt=user_prompt)

## Czy możemy zrobić to jeszcze lepiej?

Tak - dzięki trybowi JSON.

Tryb JSON to funkcja, która zapewnia, że model generuje **poprawny JSON** zgodny z określonym schematem. Po włączeniu model ściśle przestrzega reguł formatowania JSON, co czyni go idealnym do generowania danych strukturalnych, odpowiedzi API lub integracji, w których kluczowe znaczenie ma przewidywalny wynik.

**Najważniejsze informacje:**

*   Gwarantuje **poprawnie sformatowany JSON** (bez dodatkowego tekstu i nieprawidłowej składni).
*   Działa z **response_format={"type": "json_object"}** w API.
*   Jest przydatny w zadaniach takich jak strukturalne wyodrębnianie danych, generowanie konfiguracji lub przekazywanie danych między systemami.
*   Zmniejsza potrzebę post-processingu i liczbę błędów parsowania w porównaniu z dowolnym tekstem wyjściowym.

In [ ]:
def print_response(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),
        messages = [{"role" : "system", "content" : system_prompt}, {"role" : "user", "content" : user_prompt}],
        response_format={"type": "json_object"}

    )
    print(response.choices[0].message.content)

print_response(system_prompt=system_prompt, user_prompt=user_prompt)

### Ograniczenia trybu JSON:

In [ ]:
system_prompt ="""Wyodrębnij dane z podanego tekstu"""
print_response(system_prompt=system_prompt, user_prompt=user_prompt)


In [ ]:

system_prompt ="""Wyodrębnij dane z podanego tekstu w formacie JSON"""
print_response(system_prompt=system_prompt, user_prompt=user_prompt)

# Czy możemy zrobić to jeszcze lepiej?

Oczekiwane uzupełnienie:
  ```
  {
      "reason": "Lost card",
      "classified_reason": "lost_card",
      "name": "Mateo Gomez",
      "ssn": "123-45-6789",
      "dob": "09/09/1989"
  }
  ```



  ## Strukturalne odpowiedzi (structured outputs)
  
  Strukturalne odpowiedzi są rozwinięciem trybu JSON. Chociaż oba rozwiązania zapewniają generowanie poprawnego JSON, tylko Strukturalne odpowiedzi gwarantują zgodność ze schematem. Zarówno Strukturalne odpowiedzi, jak i tryb JSON są obsługiwane w Responses API, Chat Completions API, Assistants API, Fine-tuning API oraz Batch API.

Zalecamy zawsze używać Strukturalnych odpowiedzi zamiast trybu JSON, gdy tylko jest to możliwe. Pomoże to zabezpieczyć integrację na przyszłość i ułatwi pracę z API.

| Funkcja              | Strukturalne odpowiedzi                                  | Tryb JSON                                  |
|-----------------------|---------------------------------------------------------|--------------------------------------------|
| Zwraca poprawny JSON  | Tak                                                     | Tak                                        |
| Zgodność ze schematem | Tak (zobacz obsługiwane schematy)                        | Nie                                        |
| Kompatybilne modele   | gpt-4o-mini, gpt-4o-2024-08-06 i nowsze                 | gpt-3.5-turbo, gpt-4-* oraz gpt-4o-*       |
| Włączenie             | `text: { format: { type: "json_schema", "strict": true, "schema": ... } }` | `text: { format: { type: "json_object" } }` |

Więcej informacji znajdziesz w dokumentacji [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs?example=structured-data) oraz w [Structured Outputs Cookbook](https://cookbook.openai.com/examples/structured_outputs_intro).

In [ ]:
schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "my_schema",
        "schema": {
            "type": "object",
            "properties": {
                "reason": {
                    "type": "string",
                    "description": "Powód prośby"
                },
                "classified_reason": {
                    "type": "string",
                    "description": "Sklasyfikowany kod powodu może przyjmować jedną z wartości: 'lost_card', 'account_closure', 'address_change' lub 'unknown'"
                },
                "name": {
                    "type": "string"
                },
                "ssn": {
                    "type": "string"
                },
                "dob": {
                    "type": "string"
                }
            },
            "required": ["reason", "classified_reason", "name", "ssn", "dob"],
        }
    }
}

Użyj schematu, aby walidować dane JSON za pomocą odpowiedzi strukturalnych. 

In [ ]:
def print_response(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),        
        messages = [{"role" : "system", "content" : system_prompt}, {"role" : "user", "content" : user_prompt}],
        response_format=schema

    )
    print(response.choices[0].message.content)
    print(response.usage)
system_prompt ="""Wyodrębnij dane z podanego tekstu"""
user_prompt =""" 
  Cześć, nazywam się Mateo Gomez. Zgubiłem kartę kredytową 17 sierpnia i chciałbym poprosić o jej anulowanie. Ostatni zakup, którego dokonałem, to danie Chicken parmigiana w Contoso Restaurant, położonej w pobliżu Hollywood Museum, za 40 dolarów. Poniżej podaję moje dane osobowe do weryfikacji:
  Zawód: księgowy
  Numer ubezpieczenia społecznego: 123-45-6789
  Data urodzenia: 9-9-1989
  Numer telefonu: 949-555-0110
  Adres zamieszkania: 1234 Hollywood Boulevard Los Angeles CA
  Powiązane konto e-mail: mateo@contosorestaurant.com
  Kod SWIFT: CHASUS33XXX
  """
print_response(system_prompt=system_prompt, user_prompt=user_prompt)

Oprócz obsługi JSON Schema w REST API, zestawy SDK OpenAI dla Pythona i JavaScript ułatwiają także definiowanie schematów obiektów odpowiednio z użyciem Pydantic i Zod.
Nowa wersja SDK wprowadza pomocniczą funkcję **parse**, dzięki której można przekazać własny model Pydantic zamiast ręcznie definiować schemat JSON. **Zalecamy używanie tej metody, jeśli to możliwe**.

In [ ]:
from pydantic import BaseModel, Field

class CustomerRequest(BaseModel):
    reason: str = Field(description="Powód prośby")
    classified_reason: str = Field(
        description="Sklasyfikowany kod powodu może przyjmować jedną z wartości: 'lost_card', 'account_closure', 'address_change' lub 'unknown'"
    )
    name: str
    ssn: str
    dob: str

response = client.chat.completions.parse(
        model = os.getenv("AZURE_OPENAI_COMPLETION_MODEL"),        
        messages = [{"role" : "system", "content" : system_prompt}, {"role" : "user", "content" : user_prompt}],
        response_format=CustomerRequest
    )
print(response.choices[0].message.content)

In [ ]:
print(response.usage)